# Improved Industrial Defect Classification Notebook
This notebook implements improved preprocessing, SMOTE balancing, PCA for FFT features, and advanced LightGBM tuning.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df = pd.read_csv('Industrial_fault_detection.csv').dropna()
X = df.drop('Fault_Type', axis=1)
y = df['Fault_Type']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

In [ ]:
# PCA only on FFT features
fft_cols = [c for c in X.columns if 'FFT' in c]
pca = PCA(n_components=10)
X_train_fft = pca.fit_transform(X_train_bal[:, [X.columns.get_loc(c) for c in fft_cols]])
X_test_fft = pca.transform(X_test_scaled[:, [X.columns.get_loc(c) for c in fft_cols]])

In [ ]:
# Combine FFT PCA with non-FFT features
non_fft_cols = [c for c in X.columns if c not in fft_cols]
X_train_nonfft = X_train_bal[:, [X.columns.get_loc(c) for c in non_fft_cols]]
X_test_nonfft = X_test_scaled[:, [X.columns.get_loc(c) for c in non_fft_cols]]
import numpy as np
X_train_final = np.hstack([X_train_nonfft, X_train_fft])
X_test_final = np.hstack([X_test_nonfft, X_test_fft])

In [ ]:
param_dist = {
 'num_leaves': [31,63,127,255],
 'max_depth': [-1,10,20,30],
 'learning_rate': [0.1,0.05,0.01,0.005],
 'n_estimators': [200,500,1000],
 'min_child_samples': [5,10,20,50],
 'feature_fraction': [0.6,0.8,1.0],
 'bagging_fraction': [0.5,0.7,1.0],
}
model = lgb.LGBMClassifier(class_weight='balanced', random_state=42)
search = RandomizedSearchCV(model, param_dist, n_iter=20, scoring='f1_macro', cv=5, n_jobs=-1)
search.fit(X_train_final, y_train_bal)
best_model = search.best_estimator_

In [ ]:
y_pred = best_model.predict(X_test_final)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))